In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound
import os
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=1)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d')
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')
timestamp_run = peru_time.strftime('%d_%m_%Y_%H%M%S')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")


--- Fecha de inicio: 2026-05-13 ---
--- Fecha de Fin: 2026-05-14 ---
--- Fecha del proceso: 2026-05-14 00:00:00 ---


In [ ]:
## Variables de fecha como DataEntry
#var_fecha_ini = '2026-07-01'
#var_fecha_fin = '2026-07-20'
#fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio 1: {var_fecha_ini} ---")
print(f"--- Fecha de Fin 2: {var_fecha_fin} ---")
print(f"--- Fecha del proceso 3: {fecha_fin_dt} ---")

--- Fecha de inicio 1: 2026-05-13 ---
--- Fecha de Fin 2: 2026-05-14 ---
--- Fecha del proceso 3: 2026-05-14 00:00:00 ---


In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')
var_mes = fecha_fin_dt.strftime('%m')
var_fecha_file = datetime.now().strftime('%d_%m_%Y')

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")

--- Año del proceso 2026 ---
--- Mes del proceso: 05 ---
--- Fecha del archivo: 14_05_2026 ---


In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Data/Experiencia_al_Cliente/NPS_Canales/{var_anho}/{var_mes}/"
nombre_final = f"BSAC_{var_fecha_file}.csv"  # <-- CSV
proyecto_storage = "prd-izipay-data-storage-pv"
proyecto_sensitivo = "prd-izipay-data-sensitive"

In [ ]:
# =============================================================================
# 4. Creación de Tabla Temporal en BigQuery
# =============================================================================

temp_table_id = f"prd-izipay-data-operation.master_stage_financial.temp_case_sac_{var_fecha_file}"

query_temp = f"""
CREATE OR REPLACE TABLE `{temp_table_id}` AS
with segmentacion_actual as(
  select *
    from {proyecto_storage}.raw_dataentry_planeamiento.segmentacion
    where periodo = (select max(periodo)
                      from {proyecto_storage}.raw_dataentry_planeamiento.segmentacion)
)
,base_ggee as(
    SELECT
      right(trim(c.gsc_codigo_de_comercio__c),7) AS GSC_Shops__rXxXName
      ,CASE
        WHEN a.ip4i_razon_social__c IS NOT NULL THEN  a.ip4i_razon_social__c
        ELSE AEAD.DECRYPT_STRING(e.key, mcom.razon_social, e.constant)
        END AS GSC_Shops__rXxXIP4i_Raz_n_Soci__c
      ,trim(AEAD.DECRYPT_STRING(m.key,a.name, m.constant)) AS AccountXxXName
      ,trim(dc.document_number) AS AccountXxXIP4i_Nro_RUC__c
      ,trim(AEAD.DECRYPT_STRING(e.key,ow.name, e.constant)) AS AccountXxXOwnerXxXName
      ,trim(seg.banca) AS AccountXxXIZI_bancaIBK__c
      ,a.izi_volumenretail__c AS AccountXxXIZI_volumenRetail__c
      ,trim(AEAD.DECRYPT_STRING(e.key,ecs.name, e.constant)) AS AccountXxXIZI_Ejecutivo_Customer_Success__rXxXName
      ,CASE
        WHEN seg.segmento in ('BE','BI','BC') THEN 'CORPORACIONES'
        WHEN seg.segmento = 'BPE' THEN 'NEGOCIOS'
        WHEN seg.segmento = 'RETAIL' THEN 'RETAIL'
        ELSE 'SIN SEGMENTO' END AS GSC_AccountSegment__c
      ,trim(c.casenumber) AS CaseNumber
      ,trim(c.subject) AS Subject
      ,trim(c.izis_type__c) AS IZIS_Type__c
      ,trim(c.gsc_attentionlevel__c) AS GSC_AttentionLevel__c
      ,trim(c.estado_emailtocase__c) AS Estado_EmailToCase__c
      ,trim(c.origin) AS Origin
      ,trim(c.status) AS Status
      ,trim(c.motivo_de_consulta_izipay__c) AS Motivo_de_consulta_IZIPAY__c
      ,trim(c.servicio_de_comercio__c) AS Servicio_de_comercio__c
      --,trim(c.createddate) AS CreatedDate
      --,trim(c.lastmodifieddate) AS LastModifiedDate
      ,format_timestamp('%d/%m/%Y %H:%M:%S',timestamp(trim(c.createddate)),'America/Lima') as CreatedDate
      ,format_timestamp('%d/%m/%Y %H:%M:%S',timestamp(trim(c.lastmodifieddate)),'America/Lima') as LastModifiedDate
      ,trim(AEAD.DECRYPT_STRING(e.key,ou.name, e.constant)) AS OwnerXxXName
      ,trim(c.izi_unidad_de_negocio_del_propietario__c) AS IZI_unidad_de_negocio_del_propietario__c
      ,trim(c.contactid) AS ContactId
      ,trim(AEAD.DECRYPT_STRING(l.key,c.contactemail, l.constant)) AS ContactXxXEmail
      ,trim(c.izis_npsmail__c) AS Email
      ,trim(AEAD.DECRYPT_STRING(e.key,cby.name, e.constant)) AS CreatedByXxXName
      ,trim(cby.department) as CreatedByXxXDepartment
      ,trim(cby.CompanyName) as CreatedByXxXCompanyName
      ,trim(dep.name) as IZI_lookDepartamento__c
      ,trim(prov.name) as IZI_lookProvincia__c
      ,trim(dis.name) as IZI_lookDistrito__c
      ,trim(mcom.nom_comercio) as IZIS_NombreComercio__c
      ,trim(tc.asunto) as GSC_PrincipalSubject__c
      ,format_timestamp('%d/%m/%Y %H:%M:%S',timestamp(c.closeddate),'America/Lima') as closeddate
      ,trim(AEAD.DECRYPT_STRING(l.key,d.suppliedemail, l.constant)) as SuppliedEmail
    FROM {proyecto_storage}.raw_salesforce.case c
    LEFT JOIN {proyecto_storage}.raw_salesforce.case d ON trim(d.id) = trim(c.ParentId)
    LEFT JOIN {proyecto_storage}.raw_salesforce.account a ON trim(c.accountid) = trim(a.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.user ow ON trim(a.ownerid) = trim(ow.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.user ecs ON trim(a.izi_ejecutivo_customer_success__c) = trim(ecs.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.user ou ON trim(c.ownerid) = trim(ou.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.user cby ON trim(c.createdbyid) = trim(cby.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.contactpointaddress cpa on trim(cpa.id) = trim(c.gsc_contact_point_address__c)
    LEFT JOIN {proyecto_storage}.raw_salesforce.ip4i_departamento__c dep on trim(cpa.ip4i_departamento__c) = trim(dep.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.ip4i_distritos__c dis on trim(cpa.ip4i_distritos__c) = trim(dis.id)
    LEFT JOIN {proyecto_storage}.raw_salesforce.ip4i_provincia__c prov on trim(cpa.ip4i_provincia__c) = trim(prov.id)
    LEFT JOIN {proyecto_storage}.master_party.m_comercio mcom ON right(trim(c.gsc_codigo_de_comercio__c),7) = trim(mcom.cod_comercio)
    LEFT JOIN {proyecto_storage}.raw_dataentry_cx.sf_tipo_caso tc on c.gsc_principalsubject__c = tc.gsc_principalsubject__c
    LEFT JOIN prd-izipay-data-sensitive.secure_secrets.config_protected_data e ON (1=1 and e.code = 'C_FULL_NAME')
    LEFT JOIN prd-izipay-data-sensitive.secure_secrets.config_protected_data m ON (1=1 and m.code = 'C_LAST_NAME')
    LEFT JOIN prd-izipay-data-sensitive.secure_secrets.config_protected_data l ON (1=1 and l.code = 'C_EMAIL')
    LEFT JOIN segmentacion_actual seg ON right(trim(c.gsc_codigo_de_comercio__c),7) = trim(seg.codigo)
    LEFT JOIN prd-izipay-data-sensitive.master_pii.iden_party_data_control dc ON a.party_id_izi = dc.party_id_izi
    WHERE trim(c.izi_unidad_de_negocio_del_propietario__c) IN ('Soporte Tecnico Izipay', 'Soporte Comercial Izipay', 'Ecommerce','Actualización de Datos', 'Agente Izipay', 'Retiro Inmediato', 'Soporte de Cambios', 'Arisale', 'Cajero Corresponsal')
    and c.status = 'Closed'
    and c.origin in ('email2case', 'Correo electrónico', 'Ejecutivo por correo','email2caseSC', 'email2caseGE', 'email2caseAC', 'Correo electronico')
    and date(TIMESTAMP_SUB(TIMESTAMP(PARSE_TIMESTAMP('%Y-%m-%d %H:%M:%S+00', c.closeddate)),INTERVAL 5 HOUR)) = DATE '{var_fecha_ini}'
    and trim(AEAD.DECRYPT_STRING(e.key,cby.name, e.constant)) != 'Platform Integration User'
    and trim(tc.asunto) != 'Transferencias y Llamadas Vicio'
)select
distinct
  *
from base_ggee
where lower(trim(SuppliedEmail)) not like '%@izipay.pe%'
and lower(trim(SuppliedEmail)) not like '%@covisian.com%'
order by 1 asc
"""

print(f"Creando tabla temporal: {temp_table_id} ...")
clientBQ.query(query_temp).result()
print("✅ Tabla temporal creada correctamente.")
print(f"Query Ejecutado:  ...")
print(f"{query_temp} ")

Creando tabla temporal: prd-izipay-data-operation.master_stage_financial.temp_case_sac_14_05_2026 ...
✅ Tabla temporal creada correctamente.
Query Ejecutado:  ...

CREATE OR REPLACE TABLE `prd-izipay-data-operation.master_stage_financial.temp_case_sac_14_05_2026` AS
with segmentacion_actual as(
  select *
    from prd-izipay-data-storage-pv.raw_dataentry_planeamiento.segmentacion
    where periodo = (select max(periodo)
                      from prd-izipay-data-storage-pv.raw_dataentry_planeamiento.segmentacion)
)
,base_ggee as(
    SELECT
      right(trim(c.gsc_codigo_de_comercio__c),7) AS GSC_Shops__rXxXName
      ,CASE
        WHEN a.ip4i_razon_social__c IS NOT NULL THEN  a.ip4i_razon_social__c
        ELSE AEAD.DECRYPT_STRING(e.key, mcom.razon_social, e.constant)
        END AS GSC_Shops__rXxXIP4i_Raz_n_Soci__c
      ,trim(AEAD.DECRYPT_STRING(m.key,a.name, m.constant)) AS AccountXxXName
      ,trim(dc.document_number) AS AccountXxXIP4i_Nro_RUC__c
      ,trim(AEAD.DECRYPT_STRING(e.

In [ ]:
# =============================================================================
# 5. Exportación desde Tabla Temporal a GCS (CSV)
# =============================================================================
import time

uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{timestamp_run}_*.csv"
print(f"Exportando desde tabla temporal a: {uri_temporal} ...")

query_export = f"""
EXPORT DATA OPTIONS (
  uri      = '{uri_temporal}',
  format   = 'CSV',
  overwrite = true,
  header   = true
) AS
SELECT * FROM `{temp_table_id}`;
"""

# Polling con feedback cada 15s
job = clientBQ.query(query_export)
print(f"📋 Job iniciado: {job.job_id}")
t0 = time.time()

while True:
    job.reload()
    elapsed = int(time.time() - t0)
    mins, secs = divmod(elapsed, 60)
    if job.state == 'DONE':
        break
    print(f"   ⏳ {job.state} ... {mins}m {secs:02d}s transcurridos")
    time.sleep(15)

if job.errors:
    raise RuntimeError(f"❌ Error en EXPORT DATA: {job.errors}")

elapsed = int(time.time() - t0)
mins, secs = divmod(elapsed, 60)
print(f"✅ Exportación a GCS completada en {mins}m {secs:02d}s")

# Limpiar tabla temporal
clientBQ.delete_table(temp_table_id)
print(f"🧹 Tabla temporal eliminada: {temp_table_id}")


Exportando desde tabla temporal a: gs://adls-reportes/Data/Experiencia_al_Cliente/NPS_Canales/2026/05/temp_14_05_2026_*.csv ...
📋 Job iniciado: 24f965e4-3a2e-4f73-a22b-386b2b237503
   ⏳ RUNNING ... 0m 00s transcurridos
✅ Exportación a GCS completada en 0m 15s
🧹 Tabla temporal eliminada: prd-izipay-data-operation.master_stage_financial.temp_case_sac_14_05_2026


In [ ]:
# =============================================================================
# 6. Consolidación incremental a CSV comprimido (sin cargar todo en memoria)
# =============================================================================
# Escribe part a part en un archivo local temporal y luego sube a GCS.
# Así evitamos acumular todo el dataset en RAM.

print("Consolidando archivos CSV en uno solo (modo incremental)...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{timestamp_run}_"

blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar.")
else:
    local_tmp = f"/tmp/{nombre_final}"
    primera_parte = True

    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        df_part = pd.read_csv(uri_parte)
        # ── Renombrar cabeceras: XxX → . ──────────────────────────────────
        df_part.rename(columns=lambda c: c.replace('XxX', '.'), inplace=True)
        df_part.to_csv(
            local_tmp,
            mode='a',
            index=False,
            header=primera_parte,
            sep=';'
        )
        primera_parte = False
        del df_part

    # Subir archivo consolidado a GCS
    ruta_final_full = f"{ruta_base}{nombre_final}"
    blob_final = bucket.blob(ruta_final_full)
    blob_final.upload_from_filename(local_tmp)
    os.remove(local_tmp)

    # Borrar parts temporales de GCS
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: gs://{bucket_name}/{ruta_final_full}")
    print(f"🧹 {len(blobs)} archivos temporales eliminados")


Consolidando archivos CSV en uno solo (modo incremental)...
✅ ÉXITO: Archivo único creado en: gs://adls-reportes/Data/Experiencia_al_Cliente/NPS_Canales/2026/05/BSAC_14_05_2026.csv
🧹 1 archivos temporales eliminados
